In [36]:
!pip install -q langchain langchain-community langchain-core pypdf sentence-transformers qdrant-client fastapi uvicorn python-multipart python-dotenv rank-bm25 ollama transformers torch datasets peft trl FlagEmbedding pandas numpy scikit-learn plotly tqdm

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.8 MB/s eta 0:00:00


In [11]:
from google.colab import drive
drive.mount('/content/drive')

from langchain_community.document_loaders import PyPDFDirectoryLoader

PDF_PATH = "/content/drive/MyDrive/Rag_ai_tutorial/data"

loader = PyPDFDirectoryLoader(PDF_PATH)

documents = loader.load()

print("="*60)
print("TOTAL PAGES:", len(documents))
print("="*60)

print(documents[0].page_content[:500])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TOTAL PAGES: 6688



In [12]:
chunks = []

CHUNK_SIZE = 1000
OVERLAP = 200

for doc in documents:

    text = doc.page_content

    start = 0

    while start < len(text):

        end = start + CHUNK_SIZE

        chunk = text[start:end]

        if len(chunk.strip()) > 50:
            chunks.append(chunk)

        start += CHUNK_SIZE - OVERLAP

print("="*60)
print("TOTAL CHUNKS:", len(chunks))
print("="*60)

print(chunks[0][:300])

TOTAL CHUNKS: 20324
University Physics 
Volume 1 
SENIOR CONTRIBUTING AUTHORS 
WILLIAM MOEBS, FORMERLY OF LOYOLA MARYMOUNT UNIVERSITY 
SAMUEL J. LING, TRUMAN STATE UNIVERSITY 
JEFF SANNY, LOYOLA MARYMOUNT UNIVERSITY


In [13]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("Embedding Model Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


In [14]:
test_embeddings = embedding_model.encode(
    chunks[:100],
    show_progress_bar=True
)

print("Embeddings Created:", len(test_embeddings))
print("Vector Size:", len(test_embeddings[0]))

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embeddings Created: 100
Vector Size: 384


In [15]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance
from qdrant_client.models import VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="educational_tutor",
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

print("Qdrant Collection Created")

Qdrant Collection Created


In [16]:
from qdrant_client.models import PointStruct

points = []

for i, (chunk, emb) in enumerate(
        zip(chunks[:100], test_embeddings)):

    points.append(
        PointStruct(
            id=i,
            vector=emb.tolist(),
            payload={
                "text": chunk
            }
        )
    )

client.upsert(
    collection_name="educational_tutor",
    points=points
)

print("Inserted:", len(points))

Inserted: 100


In [22]:
query_vector = embedding_model.encode(
    "What is Newton's Second Law?"
)

results = client.query_points(
    collection_name="educational_tutor",
    query=query_vector.tolist(),
    limit=5
)

print(results)

points=[ScoredPoint(id=11, version=0, score=0.7301975891617374, payload={'text': "4.5 Relative Motion in One and Two Dimensions 177 \nChapter Review 182 \nCHAP TER 5 \nNewton's Laws of Motion 193 \nIntroduction 193 \n5.1 Forces 194 \n5.2 Newton's First Law 198 \n5.3 Newton's Second Law 202 \n5.4 Mass and Weight 212 \n5.5 Newton’s Third Law 214 \n5.6 Common Forces 220 \n5.7 Drawing Free-Body Diagrams 230 \nChapter Review 235 \nCHAP TER 6 \nApplications of Newton's Laws 249 \nIntroduction 249 \n6.1 Solving Problems with Newton’s Laws 249 \n6.2 Friction 265 \n6.3 Centripetal Force 276 \n6.4 Drag Force and Terminal Speed 284 \nChapter Review 293 \nCHAP TER 7 \nWork and Kinetic Energy 311 \nIntroduction 311 \n7.1 Work 312 \n7.2 Kinetic Energy 320 \n7.3 Work-Energy Theorem 323 \n7.4 Power 328 \nChapter Review 331 \nCHAP TER 8 \nPotential Energy and Conservation of Energy 341 \nIntroduction 341 \n8.1 Potential Energy of a System 341 \n8.2 Conservative and Non-Conservative Forces 350 \n8.3 Con

In [23]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

print("Phi-3 Loaded")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Phi-3 Loaded


In [35]:
question = "explain about force"

query_vector = embedding_model.encode(question)

results = client.query_points(
    collection_name="educational_tutor",
    query=query_vector.tolist(),
    limit=5
)

context = "\n\n".join([
    p.payload["text"]
    for p in results.points
])

print(context[:2000])

FIGURE 5.6 (a) The forces acting on the student are due to the chair, the table, the floor, and Earth’s gravitational attraction. (b) In solving a 
problem involving the student, we may want to consider only the forces acting along the line running through his torso. A free-body diagram 
for this situation is shown. 
In most situations, forces are grouped into two categories: contact forces and field forces. As you might guess, 
contact forces are due to direct physical contact between objects. For example, the student in Figure 5.6 
experiences the contact forces , , and , which are exerted by the chair on his posterior, the floor on his feet, and 
the table on his forearms, respectively. Field forces, however, act without the necessity of physical contact between 
objects. They depend on the presence of a “field” in the region of space surrounding the body under consideration. 
Since the student is in Earth’s gravitational field, he feels a gravitational force ; in other words, he ha

In [27]:
prompt = f"""
You are an expert educational tutor.

Use ONLY the context below.

Context:
{context}

Question:
{question}

Provide:
1. Definition
2. Formula
3. Simple Example

Answer:
"""

In [29]:
import torch

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=3000
)

# Move inputs to same device as model
inputs = {k: v.to(llm.device) for k, v in inputs.items()}

outputs = llm.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.3,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)


You are an expert educational tutor.

Use ONLY the context below.

Context:
4.5 Relative Motion in One and Two Dimensions 177 
Chapter Review 182 
CHAP TER 5 
Newton's Laws of Motion 193 
Introduction 193 
5.1 Forces 194 
5.2 Newton's First Law 198 
5.3 Newton's Second Law 202 
5.4 Mass and Weight 212 
5.5 Newton’s Third Law 214 
5.6 Common Forces 220 
5.7 Drawing Free-Body Diagrams 230 
Chapter Review 235 
CHAP TER 6 
Applications of Newton's Laws 249 
Introduction 249 
6.1 Solving Problems with Newton’s Laws 249 
6.2 Friction 265 
6.3 Centripetal Force 276 
6.4 Drag Force and Terminal Speed 284 
Chapter Review 293 
CHAP TER 7 
Work and Kinetic Energy 311 
Introduction 311 
7.1 Work 312 
7.2 Kinetic Energy 320 
7.3 Work-Energy Theorem 323 
7.4 Power 328 
Chapter Review 331 
CHAP TER 8 
Potential Energy and Conservation of Energy 341 
Introduction 341 
8.1 Potential Energy of a System 341 
8.2 Conservative and Non-Conservative Forces 350 
8.3 Conservation of Energy 353 
8.4 Potential 

In [30]:
BATCH_SIZE = 100

for start in range(0, len(chunks), BATCH_SIZE):

    batch_chunks = chunks[start:start+BATCH_SIZE]

    batch_vectors = embedding_model.encode(
        batch_chunks,
        show_progress_bar=False
    )

    points = []

    for idx, (text, vector) in enumerate(
        zip(batch_chunks, batch_vectors)
    ):

        points.append(
            PointStruct(
                id=start + idx,
                vector=vector.tolist(),
                payload={
                    "text": text
                }
            )
        )

    client.upsert(
        collection_name="educational_tutor",
        points=points
    )

    print(
        f"Uploaded {min(start+BATCH_SIZE,len(chunks))}/{len(chunks)}"
    )

Uploaded 100/20324
Uploaded 200/20324
Uploaded 300/20324
Uploaded 400/20324
Uploaded 500/20324
Uploaded 600/20324
Uploaded 700/20324
Uploaded 800/20324
Uploaded 900/20324
Uploaded 1000/20324
Uploaded 1100/20324
Uploaded 1200/20324
Uploaded 1300/20324
Uploaded 1400/20324
Uploaded 1500/20324
Uploaded 1600/20324
Uploaded 1700/20324
Uploaded 1800/20324
Uploaded 1900/20324
Uploaded 2000/20324
Uploaded 2100/20324
Uploaded 2200/20324
Uploaded 2300/20324
Uploaded 2400/20324
Uploaded 2500/20324
Uploaded 2600/20324
Uploaded 2700/20324
Uploaded 2800/20324
Uploaded 2900/20324
Uploaded 3000/20324
Uploaded 3100/20324
Uploaded 3200/20324
Uploaded 3300/20324
Uploaded 3400/20324
Uploaded 3500/20324
Uploaded 3600/20324
Uploaded 3700/20324
Uploaded 3800/20324
Uploaded 3900/20324
Uploaded 4000/20324
Uploaded 4100/20324
Uploaded 4200/20324
Uploaded 4300/20324
Uploaded 4400/20324
Uploaded 4500/20324
Uploaded 4600/20324
Uploaded 4700/20324
Uploaded 4800/20324
Uploaded 4900/20324
Uploaded 5000/20324
Uploaded 

/tmp/ipykernel_332/3110509116.py:28: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20100 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(


Uploaded 20100/20324
Uploaded 20200/20324
Uploaded 20300/20324
Uploaded 20324/20324


In [38]:
import pickle

with open(
    "/content/drive/MyDrive/Rag_ai_tutorial/chunks.pkl",
    "wb"
) as f:
    pickle.dump(chunks, f)

print("Chunks Saved")

Chunks Saved


In [40]:
all_embeddings = embedding_model.encode(
    chunks,
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/636 [00:00<?, ?it/s]

In [41]:
import pickle

with open(
    "/content/drive/MyDrive/Rag_ai_tutorial/embeddings.pkl",
    "wb"
) as f:
    pickle.dump(all_embeddings, f)

print("Embeddings Saved")

Embeddings Saved


In [42]:
from qdrant_client import QdrantClient

client = QdrantClient(
    path="/content/drive/MyDrive/Rag_ai_tutorial/vector_db"
)

In [ ]:
llm.save_pretrained(
    "/content/drive/MyDrive/Rag_ai_tutorial/phi3_model"
)

tokenizer.save_pretrained(
    "/content/drive/MyDrive/Rag_ai_tutorial/phi3_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]